In [4]:
from dotenv import load_dotenv
import anthropic, json, pandas as pd
from pathlib import Path
import os

In [5]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [8]:
df = pd.read_csv(root / "csv_cleaned/enunciados_ejercicios_clean.csv", encoding="utf-8")

In [12]:
mates_df = df[df.asignatura.isin(["Mates CCSS", "Mates II"])]

In [17]:
mates_df

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo
486,mates_ccss_programacion_lineal_2001_junio_ejer...,Mates CCSS,Mates CCSS,2001,Junio,Ejercicio 1 opción A,Programación Lineal,SOCIALES II. 2001 JUNIO. EJERCICIO 1 OPCIÓN A.,"Para fabricar dos tipos de cable, A y B, que s...",2001 - Programación Lineal.pdf
487,mates_ccss_matrices_y_determinantes_2001_junio...,Mates CCSS,Mates CCSS,2001,Junio,"Ejercicio 1, opción B",Matrices Y Determinantes,SOCIALES II. 2001. JUNIO. EJERCICIO 1. OPCIÓN B,a) Determine los valores de x e y que hacen ci...,2001 - Matrices Y Determinantes.pdf
488,mates_ccss_funciones_ccss_2001_junio_ejercicio...,Mates CCSS,Mates CCSS,2001,Junio,"Ejercicio 2, opción A",Funciones CCSS,SOCIALES II. 2001. JUNIO. EJERCICIO 2. OPCIÓN A,Calcule las funciones derivadas de las siguien...,2001 - Funciones CCSS.pdf
489,mates_ccss_inferencia_estadistica_2001_junio_e...,Mates CCSS,Mates CCSS,2001,Junio,Ejercicio 3 Parte Ii opción A,Inferencia Estadística,SOCIALES II. 2001 JUNIO. EJERCICIO 3 PARTE II ...,Un estudio realizado sobre 100 usuarios revela...,2001 - Inferencia Estadística.pdf
490,mates_ccss_inferencia_estadistica_2001_junio_e...,Mates CCSS,Mates CCSS,2001,Junio,Ejercicio 3 Parte Ii opción B,Inferencia Estadística,SOCIALES II. 2001 JUNIO. EJERCICIO 3 PARTE II ...,La cantidad de hemoglobina en sangre del hombr...,2001 - Inferencia Estadística.pdf
...,...,...,...,...,...,...,...,...,...,...
2961,mates_ii_matrices_y_determinantes_2026_junio_e...,Mates II,Mates II,2026,Junio,"Ejercicio 3,2",Matrices Y Determinantes,MATEMÁTICAS II. 2026. JUNIO. EJERCICIO 3.2,Considera la matriz A =\n \n2\n3\n\n\n...,2026 - Matrices Y Determinantes.pdf
2962,mates_ii_integrales_2026_junio_ejercicio_4_1,Mates II,Mates II,2026,Junio,"Ejercicio 4,1",Integrales,MATEMÁTICAS II. 2026. JUNIO. EJERCICIO 4.1,Dada la función f : → definida por: f ( x ) =\...,2026 - Integrales.pdf
2963,mates_ii_probabilidad_2026_junio_ejercicio_4_2,Mates II,Mates II,2026,Junio,"Ejercicio 4,2",Probabilidad,MATEMÁTICAS II. 2026 JUNIO. EJERCICIO 4.2,Disponemos de dos bolsas con monedas de oro y ...,2026 - Probabilidad.pdf
2964,mates_ii_distribucion_normal_y_binomial_2026_r...,Mates II,Mates II,2026,Reserva 1,"Ejercicio 4,2",Distribución normal y binomial,MATEMÁTICAS II. 2026. RESERVA 1. EJERCICIO 4.2,Las tallas de las niñas al nacer se distribuye...,2026 - Distribución normal y binomial.pdf


In [20]:
matrices = mates_df.loc[mates_df.tema == "Matrices Y Determinantes", ["id", "asignatura", "enunciado"]]

In [21]:
matrices

,id,asignatura,enunciado
487,mates_ccss_matrices_y_determinantes_2001_junio...,Mates CCSS,a) Determine los valores de x e y que hacen ci...
494,mates_ccss_matrices_y_determinantes_2001_reser...,Mates CCSS, 1 x 1\n \nSe considera la matriz A 1 1 ...
501,mates_ccss_matrices_y_determinantes_2001_reser...,Mates CCSS,Resuelva la siguiente ecuación matricial: AX ...
508,mates_ccss_matrices_y_determinantes_2001_reser...,Mates CCSS,1 0 0 0 1\n   \nSiendoA 2 1 0 y B  1...
536,mates_ccss_matrices_y_determinantes_2002_reser...,Mates CCSS,3 1 1 z\n  x    \nSean las matri...
...,...,...,...
2946,mates_ii_matrices_y_determinantes_2025_reserva...,Mates II,Considera la matriz A =\n 0\n1\n− 1\n0\n...
2947,mates_ii_matrices_y_determinantes_2025_reserva...,Mates II,Sean las matrices A =\n 2\n3\n2\n4\n\ny ...
2953,mates_ii_matrices_y_determinantes_2025_reserva...,Mates II,a a a \n 11 12 13\nSe considera la matriz ...
2954,mates_ii_matrices_y_determinantes_2025_reserva...,Mates II,Sean las matrices A =\n a\nb\n3\n1\n\ny ...


In [22]:
matrices.to_csv(root / "classifier/matrices.csv", index = False, encoding="utf-8")

In [23]:
load_dotenv()
client = anthropic.Anthropic()  # lee ANTHROPIC_API_KEY del entorno

In [26]:
system_md = (root / "classifier/matrices.md").read_text(encoding="utf-8")
#df = pd.read_csv("../classifier/sample_classification.csv")
df = matrices

In [27]:
def classify(enunciado, system_md, modelo="claude-haiku-4-5-20251001"):
    resp = client.messages.create(
        model=modelo,
        max_tokens=300,
        system=[{"type": "text", "text": system_md,
                 "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": f"Enunciado:\n{enunciado}"}],
    )
    texto = resp.content[0].text.strip().removeprefix("```json").removesuffix("```").strip()
    return json.loads(texto)

In [30]:
output_file = root / "classifier/resultados.jsonl"
already_processed = {json.loads(l)["id"] for l in output_file.open()} if output_file.exists() else set()

In [31]:
for index, row in df.iterrows():
    if row.id in already_processed:
        continue
    try:
        r = classify(row.enunciado, system_md)
        r["id"] = row.id
    except Exception as e:
        r = {"id": row.id, "error": str(e)}
    with output_file.open("a", encoding="utf-8") as f:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [40]:
df = pd.read_json(root / "classifier/resultados.jsonl", lines = True)

In [45]:
df.loc[df.asignatura.isin(["Mates CCSS", "Mates II"]), "tema"] = "Matrices y determinantes"

In [46]:
df

,asignatura,tema,tipo_ejercicio,confianza,id,error
0,Física,Campo gravitatorio,plano_inclinado,alta,2022_Junio_A1,NaN
1,Física,Campo gravitatorio,plano_inclinado,alta,2024_Julio_A2,NaN
2,Física,Campo gravitatorio,satélites,alta,2024_Reserva 2_A2,NaN
3,Física,Campo gravitatorio,distribución_masas,alta,2023_Junio_A2,NaN
4,Física,Campo gravitatorio,satélites,alta,2016_Reserva 2_1_A,NaN
...,...,...,...,...,...,...
299,Mates II,Matrices y determinantes,potencias,alta,mates_ii_matrices_y_determinantes_2025_reserva...,NaN
300,Mates II,Matrices y determinantes,ecuación matricial,alta,mates_ii_matrices_y_determinantes_2025_reserva...,NaN
301,Mates CCSS,Matrices y determinantes,determinantes,media,mates_ii_matrices_y_determinantes_2025_reserva...,NaN
302,Mates CCSS,Matrices y determinantes,potencias|ecuación matricial,media,mates_ii_matrices_y_determinantes_2025_reserva...,NaN


In [48]:
df.to_csv(root / "classifier/classified_exercises.csv", index = False, encoding="utf-8")

In [34]:
df_out = (pd.read_json(output_file, lines=True)
          .sort_values("id")
)
df_out = df_out[["id"] + list(df_out.columns[:-2])]

df_out


,id,asignatura,tema,tipo_ejercicio,confianza
4,2016_Reserva 2_1_A,Física,Campo gravitatorio,satélites,alta
8,2017_Junio_1_B,Física,Campo gravitatorio,satélites,media
7,2020_Reserva 4_1,Física,Campo gravitatorio,distribución_masas,alta
6,2021_Julio_A1,Física,Campo gravitatorio,satélites,alta
9,2021_Reserva 3_A2,Física,Campo gravitatorio,satélites,alta
...,...,...,...,...,...
299,mates_ii_matrices_y_determinantes_2025_reserva...,Mates II,NaN,potencias,alta
300,mates_ii_matrices_y_determinantes_2025_reserva...,Mates II,NaN,ecuación matricial,alta
301,mates_ii_matrices_y_determinantes_2025_reserva...,Mates CCSS,NaN,determinantes,media
302,mates_ii_matrices_y_determinantes_2025_reserva...,Mates CCSS,NaN,potencias|ecuación matricial,media


In [49]:
df.columns

Index(['asignatura', 'tema', 'tipo_ejercicio', 'confianza', 'id', 'error'], dtype='str')

In [50]:
df.head()

,asignatura,tema,tipo_ejercicio,confianza,id,error
0,Física,Campo gravitatorio,plano_inclinado,alta,2022_Junio_A1,NaN
1,Física,Campo gravitatorio,plano_inclinado,alta,2024_Julio_A2,NaN
2,Física,Campo gravitatorio,satélites,alta,2024_Reserva 2_A2,NaN
3,Física,Campo gravitatorio,distribución_masas,alta,2023_Junio_A2,NaN
4,Física,Campo gravitatorio,satélites,alta,2016_Reserva 2_1_A,NaN


In [55]:
def fix_asignatura(row):
    if "mates_ccss" in row.id:
        row["asignatura"] = "Mates CCSS"
        row["tema"] = "Matrices y determinantes"
    elif "mates_ii" in row.id:
        row["asignatura"] = "Mates II"
        row["tema"] = "Matrices y determinantes"

    return row   

In [58]:
df = df.apply(fix_asignatura, axis = 1)

In [59]:
df[df.asignatura != "Física"]

,asignatura,tema,tipo_ejercicio,confianza,id,error
10,Mates CCSS,Matrices y determinantes,ecuación matricial,alta,mates_ccss_matrices_y_determinantes_2001_junio...,NaN
11,Mates CCSS,Matrices y determinantes,valores tenga inversa,alta,mates_ccss_matrices_y_determinantes_2001_reser...,NaN
12,Mates CCSS,Matrices y determinantes,ecuación matricial,alta,mates_ccss_matrices_y_determinantes_2001_reser...,NaN
13,Mates CCSS,Matrices y determinantes,ecuación matricial,alta,mates_ccss_matrices_y_determinantes_2001_reser...,NaN
14,Mates CCSS,Matrices y determinantes,ecuación matricial,alta,mates_ccss_matrices_y_determinantes_2002_reser...,NaN
...,...,...,...,...,...,...
299,Mates II,Matrices y determinantes,potencias,alta,mates_ii_matrices_y_determinantes_2025_reserva...,NaN
300,Mates II,Matrices y determinantes,ecuación matricial,alta,mates_ii_matrices_y_determinantes_2025_reserva...,NaN
301,Mates II,Matrices y determinantes,determinantes,media,mates_ii_matrices_y_determinantes_2025_reserva...,NaN
302,Mates II,Matrices y determinantes,potencias|ecuación matricial,media,mates_ii_matrices_y_determinantes_2025_reserva...,NaN


In [51]:
df_enun = pd.read_csv(root / "csv_cleaned/enunciados_ejercicios_clean.csv",
                      encoding="utf-8")

In [60]:
df_enun.head()

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo
0,fisica_campo_electrico_y_magnetico_2016_junio_...,Física,Física,2016,Junio,"Ejercicio 1, opción A",Campo eléctrico y magnético,FISICA. 2016. JUNIO. EJERCICIO 1. OPCIÓN A,a) Campo eléctrico creado por una carga puntua...,2016 - Campo eléctrico y magnético.pdf
1,fisica_campo_gravitatorio_2016_junio_ejercicio...,Física,Física,2016,Junio,"Ejercicio 1, opción B",Campo gravitatorio,FISICA. 2016. JUNIO. EJERCICIO 1. OPCIÓN B,a) Defina velocidad de escape de un planeta y ...,2016 - Campo gravitatorio.pdf
2,fisica_ondas_2016_junio_ejercicio_2_opcion_a,Física,Física,2016,Junio,"Ejercicio 2, opción A",Ondas,FISICA. 2016. JUNIO. EJERCICIO 2. OPCIÓN A,a) Explique las características cinemáticas de...,2016 - Ondas.pdf
3,fisica_fisica_cuantica_y_nuclear_2016_junio_ej...,Física,Física,2016,Junio,"Ejercicio 2, opción B",Física cuántica y nuclear,FISICA. 2016. JUNIO. EJERCICIO 2. OPCIÓN B,a) Teoría de Einstein del efecto fotoeléctrico...,2016 - Física cuántica y nuclear.pdf
4,fisica_campo_gravitatorio_2016_junio_ejercicio...,Física,Física,2016,Junio,"Ejercicio 3, opción A",Campo gravitatorio,FISICA. 2016. JUNIO. EJERCICIO 3. OPCIÓN A,Dos partículas de masas m 3kg y m  5kg se en...,2016 - Campo gravitatorio.pdf


In [64]:
pd.merge(df, df_enun, on=["id", "asignatura"]).to_csv(root / "csv_cleaned/enunciados_clasificados.csv",
                                                      index = False, encoding="utf-8")